# Landhills Winery — Optimal Blending Plan
### Prescriptive Analytics Using Linear & Mixed-Integer Optimization

This notebook implements a profit-maximization optimization model to support wine-blending decisions under supply, regulatory, quality, and commercial constraints.

The goal is not mathematical sophistication, but decision quality under real-world constraints.


## Decision Context

Landhills Winery must decide how to allocate limited grape inventories across multiple wine products while complying with regulatory, quality, and commercial constraints.

These decisions are interdependent: adjusting one blend affects feasibility and profitability elsewhere in the portfolio. As a result, heuristic or spreadsheet-based planning is unreliable.

This model provides a transparent, defensible framework for evaluating tradeoffs and identifying an optimal production plan.


In [1]:
# Core optimization and data libraries used to formulate and solve the blending model
from gurobipy import *

In [2]:
m = Model() 

Set parameter Username
Set parameter LicenseID to value 2700936
Academic license - for non-commercial use only - expires 2026-08-29


### Set up variable and constraint Python lists

In [ ]:
## Decision Variables

Decision variables represent how much of each grape type is allocated to each wine product.

Variables are named to be manager-readable, reflecting real production decisions rather than abstract indices.

In [3]:
# List of all wine blend decision variables
# vintage_cabernet_a, vintage_cabernet_b, vintage_cabernet_c = Vintage Cabernets (from different regions/years)
# N = Non-vintage Cabernet, M = Non-vintage Merlot

winetype = ['vintage_cabernet_a_blend1', 'vintage_cabernet_b_blend1', 'vintage_cabernet_c_blend1',
             'vintage_cabernet_a_blend2', 'vintage_cabernet_b_blend2', 'vintage_cabernet_c_blend2',
             'vintage_cabernet_a_blend3', 'vintage_cabernet_b_blend3', 'vintage_cabernet_c_blend3',
             'vintage_cabernet_a_blend4', 'vintage_cabernet_b_blend4', 'vintage_cabernet_c_blend4',
             'nonvintage_cabernet_blend1', 'nonvintage_cabernet_blend2', 'nonvintage_cabernet_blend3', 'nonvintage_cabernet_blend4',
             'nonvintage_merlot_blend1', 'nonvintage_merlot_blend2', 'nonvintage_merlot_blend3', 'nonvintage_merlot_blend4']

## Constraints

# Constraints are grouped into business-meaningful categories that reflect how managers reason about feasibility and tradeoffs.
# Supply constraints: ensure grape usage does not exceed available inventory
# Regulatory constraints: enforce alcohol content, varietal, and regional composition rules
# Quality constraints: maintain minimum and maximum blend proportions
# Commercial constraints: demand limits, quantity discounts, and exclusivity conditions

# Constraints with “≤” relationships
# These enforce upper bounds from product specifications or regulations (e.g., maximum sugar/acidity/alcohol, production caps).
# These represent limits (e.g., sugar, acidity, alcohol max, and quantity)

leq_constraints = ['Sugar Level 1', 'Sugar Level 2', 'Sugar Level 3', 'Sugar Level 4',
                   'Acidity 1', 'Acidity 2', 'Acidity 3', 'Acidity 4', 'Acidity 5',
                   'Alcohol 11', 'Alcohol 12', 'Alcohol 13', 'Alcohol 14', 'Alcohol 15',
                   'Quantity 1', 'Quantity 2', 'Quantity 3', 'Quantity 4']

# Constraints with “≥” relationships
# These enforce minimum requirements (e.g., minimum varietal/vintage/region percentages, minimum alcohol).
# These represent minimum requirements (e.g., grape type %, alcohol min, vintage year %, area %)

geq_constraints = ['Grape type 1', 'Grape type 2', 'Grape type 3', 'Grape type 4', 'Grape type 5',
                   'Alcohol 21', 'Alcohol 22', 'Alcohol 23', 'Alcohol 24', 'Alcohol 25',
                   'Year 1', 'Year 2', 'Year 3',
                   'Area 1', 'Area 2', 'Area 3']


### Set  objective coefficients using Python dictionary

In [ ]:
## Objective Function

The objective is to maximize total contribution margin across all wine products, accounting for pricing, production costs, and applicable quantity discounts.

This formulation ensures the solution reflects economic value, not just feasibility.

In [4]:
profit_contribution = { 'vintage_cabernet_a_blend1' : 6.65 ,
                        'vintage_cabernet_b_blend1' : 6.65 , 
                        'vintage_cabernet_c_blend1' : 6.65 , 
                        'vintage_cabernet_a_blend2' : 6.40 , 
                        'vintage_cabernet_b_blend2' : 6.40 , 
                        'vintage_cabernet_c_blend2' : 6.40 , 
                        'vintage_cabernet_a_blend3' : 6.90 ,
                        'vintage_cabernet_b_blend3' : 6.90 ,
                        'vintage_cabernet_c_blend3' : 6.90 ,
                        'vintage_cabernet_a_blend4' : 7.45 ,
                        'vintage_cabernet_b_blend4' : 7.45 ,
                        'vintage_cabernet_c_blend4' : 7.45 ,
                        'nonvintage_cabernet_blend1' : 3.15 ,
                        'nonvintage_cabernet_blend2' : 2.90 ,
                        'nonvintage_cabernet_blend3' : 3.40 ,
                        'nonvintage_cabernet_blend4' : 3.95 ,
                        'nonvintage_merlot_blend1' : 0.60 , 
                        'nonvintage_merlot_blend2' : 0.35 ,
                        'nonvintage_merlot_blend3' : 0.85 ,
                        'nonvintage_merlot_blend4' : 1.40 
                      }

### Load variables into the Gurobi model

In [5]:
blend_allocation = m.addVars(winetype, name = 'blend_allocation')

### Set the matrix of left-hand-side constraint coefficients (Python dictionary)

In [6]:
# Contraints that are <=
lhs_constr_matrix_leq = {
                         'Sugar Level 1': { 
                                           'vintage_cabernet_a_blend1': 0.0012, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0.0025, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0.0030, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0.0008 , 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Sugar Level 2': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0.0012, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0.0025, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0.0030, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0.0008 , 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Sugar Level 3': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0.0012, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0.0025, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0.0030, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0.0008 , 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Sugar Level 4': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0.0012, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0.0025, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0.0030, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0.0008 , 'nonvintage_merlot_blend4': 0
                                           },
                         'Acidity 1':     { 
                                           'vintage_cabernet_a_blend1': 0.35, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0.75, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0.55, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0.25, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Acidity 2':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0.35, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0.75, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0.55, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0.25, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Acidity 3':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0.35, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0.75, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0.55, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0.25, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Acidity 4':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0.35, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0.75, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0.55, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0.25, 'nonvintage_merlot_blend4': 0
                                           },
                         'Acidity 5':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0.35,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0.75,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0.55,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0.25
                                           },
                         'Alcohol 11':    { 
                                           'vintage_cabernet_a_blend1': 0.135, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0.153, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0.115, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0.157, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Alcohol 12':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0.135, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0.153, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0.115, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0.157, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Alcohol 13':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0.135, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0.153, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0.115, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0.157, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Alcohol 14':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0.135, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0.153, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0.115, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0.157, 'nonvintage_merlot_blend4': 0
                                           },
                         'Alcohol 15':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0.135,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0.153,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0.115,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0.157
                                           },
                         'Quantity 1':     { 
                                           'vintage_cabernet_a_blend1': 1, 'vintage_cabernet_b_blend1': 1, 'vintage_cabernet_c_blend1': 1, 'nonvintage_cabernet_blend1': 1, 'nonvintage_merlot_blend1': 1,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Quantity 2':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 1, 'vintage_cabernet_b_blend2': 1, 'vintage_cabernet_c_blend2': 1, 'nonvintage_cabernet_blend2': 1, 'nonvintage_merlot_blend2': 1,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Quantity 3':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 1, 'vintage_cabernet_b_blend3': 1, 'vintage_cabernet_c_blend3': 1, 'nonvintage_cabernet_blend3': 1, 'nonvintage_merlot_blend3': 1,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                           },
                         'Quantity 4':     { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 1, 'vintage_cabernet_b_blend4': 1, 'vintage_cabernet_c_blend4': 1, 'nonvintage_cabernet_blend4': 1, 'nonvintage_merlot_blend4': 1
                                           }
}

# Contraints that are >=
lhs_constr_matrix_geq = {
                       'Grape type 1' : { 'vintage_cabernet_a_blend1' : 1, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 1, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 1, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 2' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 1, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 1, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 1, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 3' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 1, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 1, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 1, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 4' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 1, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 1, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 1, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 5' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 1 } , 
    
                         'Alcohol 21' : { 'vintage_cabernet_a_blend1' : 0.135, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0.153, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0.115, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0.157, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
                      
                         'Alcohol 22' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0.135, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0.153, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0.115, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0.157, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                         'Alcohol 23' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0.135, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0.153, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0.115, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0.157, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                         'Alcohol 24' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0.135, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0.153, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0.115, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0.157, 'nonvintage_merlot_blend4' : 0 }, 
                        
                         'Alcohol 25' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0.135, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0.153, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0.115,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0.157 }, 
    
                             'Year 1' : { 'vintage_cabernet_a_blend1' : 1, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 1, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 

                             'Year 2' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 1, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 1, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                             'Year 3' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 1, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 1, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                             'Area 1' : { 'vintage_cabernet_a_blend1' : 1, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 1, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                             'Area 2' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 1, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 1, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                             'Area 3' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 1, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 1, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }
                        }

### Set the right-hand-side constraint vector (Python dictionary)

In [7]:
rhs_constr_vector = { 
                     'Sugar Level 1': { 
                                           'vintage_cabernet_a_blend1': 0.0020, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0.0020, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0.0020, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0.0020, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                     'Sugar Level 2': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0.0020, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0.0020, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0.0020, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0.0020, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                    'Sugar Level 3': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0.0020, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0.0020, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0.0020, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0.0020, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                    'Sugar Level 4': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0.0030, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0.0030, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0.0030, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0.0030, 'nonvintage_merlot_blend4': 0
                                       },
                        'Acidity 1': { 
                                           'vintage_cabernet_a_blend1': 0.70, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0.70, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0.70, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0.70, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                        'Acidity 2': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0.70, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0.70, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0.70, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0.70, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                        'Acidity 3': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0.70, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0.70, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0.70, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0.70, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                        'Acidity 4': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0.70, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0.70, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0.70, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0.70, 'nonvintage_merlot_blend4': 0
                                       },
                        'Acidity 5': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0.30,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0.30,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0.30,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0.30
                                       },
                        'Alcohol 11': { 
                                           'vintage_cabernet_a_blend1': 0.15, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0.15, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0.15, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0.15, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                        'Alcohol 12': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0.15, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0.15, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0.15, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0.15, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                        'Alcohol 13': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0.15, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0.15, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0.15, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0.15, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0
                                       },
                        'Alcohol 14': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0.15, 'nonvintage_merlot_blend1': 0,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0.15, 'nonvintage_merlot_blend2': 0,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0.15, 'nonvintage_merlot_blend3': 0,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0.15, 'nonvintage_merlot_blend4': 0
                                       },
                        'Alcohol 15': { 
                                           'vintage_cabernet_a_blend1': 0, 'vintage_cabernet_b_blend1': 0, 'vintage_cabernet_c_blend1': 0, 'nonvintage_cabernet_blend1': 0, 'nonvintage_merlot_blend1': 0.15,
                                           'vintage_cabernet_a_blend2': 0, 'vintage_cabernet_b_blend2': 0, 'vintage_cabernet_c_blend2': 0, 'nonvintage_cabernet_blend2': 0, 'nonvintage_merlot_blend2': 0.15,
                                           'vintage_cabernet_a_blend3': 0, 'vintage_cabernet_b_blend3': 0, 'vintage_cabernet_c_blend3': 0, 'nonvintage_cabernet_blend3': 0, 'nonvintage_merlot_blend3': 0.15,
                                           'vintage_cabernet_a_blend4': 0, 'vintage_cabernet_b_blend4': 0, 'vintage_cabernet_c_blend4': 0, 'nonvintage_cabernet_blend4': 0, 'nonvintage_merlot_blend4': 0.15
                                       },

                      'Grape type 1' : { 'vintage_cabernet_a_blend1' : 0.75, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                         'vintage_cabernet_a_blend2' : 0.75, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                         'vintage_cabernet_a_blend3' : 0.75, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                         'vintage_cabernet_a_blend4' : 0.75, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 2' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0.75, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0.75, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0.75, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0.75, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 3' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0.75, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0.75, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0.75, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0.75, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 4' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0.75, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0.75, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0.75, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0.75, 'nonvintage_merlot_blend4' : 0 } , 
    
                       'Grape type 5' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0.75, 
                                          'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0.75, 
                                          'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0.75,  
                                          'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0.75 } , 
    
                         'Alcohol 21' : { 'vintage_cabernet_a_blend1' : 0.10, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                          'vintage_cabernet_a_blend2' : 0.10, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                          'vintage_cabernet_a_blend3' : 0.10, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                          'vintage_cabernet_a_blend4' : 0.10, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
                      
                        'Alcohol 22' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0.10, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                         'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0.10, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                         'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0.10, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                         'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0.10, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                        'Alcohol 23' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0.10, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                         'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0.10, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                         'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0.10, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                         'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0.10, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                        'Alcohol 24' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0.10, 'nonvintage_merlot_blend1' : 0, 
                                         'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0.10, 'nonvintage_merlot_blend2' : 0, 
                                         'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0.10, 'nonvintage_merlot_blend3' : 0,  
                                         'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0.10, 'nonvintage_merlot_blend4' : 0 }, 
    
                        'Alcohol 25' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0.10, 
                                         'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0.10, 
                                         'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0.10,  
                                         'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0.10 }, 
    
                         'Year 1' : { 'vintage_cabernet_a_blend1' : 0.95, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                      'vintage_cabernet_a_blend2' : 0.95, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                      'vintage_cabernet_a_blend3' : 0.95, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                      'vintage_cabernet_a_blend4' : 0.95, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 

                         'Year 2' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0.95, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                      'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0.95, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                      'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0.95, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                      'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0.95, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 

                         'Year 3' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0.95, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                      'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0.95, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                      'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0.95, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                      'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0.95, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 

                         'Area 1' : { 'vintage_cabernet_a_blend1' : 0.85, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                    'vintage_cabernet_a_blend2' : 0.85, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                    'vintage_cabernet_a_blend3' : 0.85, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                    'vintage_cabernet_a_blend4' : 0.85, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                         'Area 2' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0.85, 'vintage_cabernet_c_blend1' : 0, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                    'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0.85, 'vintage_cabernet_c_blend2' : 0, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                    'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0.85, 'vintage_cabernet_c_blend3' : 0, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                    'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0.85, 'vintage_cabernet_c_blend4' : 0, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 }, 
    
                         'Area 3' : { 'vintage_cabernet_a_blend1' : 0, 'vintage_cabernet_b_blend1' : 0, 'vintage_cabernet_c_blend1' : 0.85, 'nonvintage_cabernet_blend1' : 0, 'nonvintage_merlot_blend1' : 0, 
                                    'vintage_cabernet_a_blend2' : 0, 'vintage_cabernet_b_blend2' : 0, 'vintage_cabernet_c_blend2' : 0.85, 'nonvintage_cabernet_blend2' : 0, 'nonvintage_merlot_blend2' : 0, 
                                    'vintage_cabernet_a_blend3' : 0, 'vintage_cabernet_b_blend3' : 0, 'vintage_cabernet_c_blend3' : 0.85, 'nonvintage_cabernet_blend3' : 0, 'nonvintage_merlot_blend3' : 0,  
                                    'vintage_cabernet_a_blend4' : 0, 'vintage_cabernet_b_blend4' : 0, 'vintage_cabernet_c_blend4' : 0.85, 'nonvintage_cabernet_blend4' : 0, 'nonvintage_merlot_blend4' : 0 },

                         'Quantity 1' : 50000, 
                         'Quantity 2' : 60000, 
                         'Quantity 3' : 30000, 
                         'Quantity 4' : 200000

}

### Load the constraints into Gurobi. The use of quicksum and the for loops is convenient when there are many constraints.

In [8]:
# Generate 'less than or equal to' constraints
leq_constraints_set1 = m.addConstrs(
    (
        quicksum(lhs_constr_matrix_leq[constr][wine] * blend_allocation[wine]
                 for wine in winetype) <= (
            quicksum(rhs_constr_vector[constr][wine] * blend_allocation[wine]
                     for wine in rhs_constr_vector[constr])
            if isinstance(rhs_constr_vector[constr], dict) else rhs_constr_vector[constr]
        )
        for constr in leq_constraints
    ),
    name="leq_constraints"
)

# Generate 'greater than or equal to' constraints
leq_constraints_set2 = m.addConstrs(
    (
        quicksum(lhs_constr_matrix_geq[constr][wine] * blend_allocation[wine]
                 for wine in winetype) >= (
            quicksum(rhs_constr_vector[constr][wine] * blend_allocation[wine]
                     for wine in rhs_constr_vector[constr])
            if isinstance(rhs_constr_vector[constr], dict) else rhs_constr_vector[constr]
        )
        for constr in geq_constraints
    ),
    name="geq_constraints"
)


### 

In [9]:


obj = quicksum(
   profit_contribution[wine] * blend_allocation[wine] for wine in winetype
)

m.setObjective(obj, GRB.MAXIMIZE)

m.optimize()

#Step 1.2: 
# In order to determine most valuable grape type I am looking at the quantity constraints and among the 4 quantities of grapes, Quantity 3
# has the highest shadow price of 7.85 (see below). Hence, Cabernet Sauvignon (San Luis Obispo, 2011) is the most valuable grape.

#Step 1.3: 
# Again, looking at the shadow price of all constraints, the highest shadow price with an allowable increase is Alcohol15
# Alcohol15 has a shadow price of 200, allowable increase of 511 (see below)
# Alcohol15 corresponds to the equation [0.135*nonvintage_merlot_blend1 + 0.153*nonvintage_merlot_blend2 + 0.115*nonvintage_merlot_blend3 + 0.157*nonvintage_merlot_blend4 <= 0.15*(nonvintage_merlot_blend1 + nonvintage_merlot_blend2 + nonvintage_merlot_blend3 + nonvintage_merlot_blend4)]
# An increase of 1% would mean multiplying the RHS by 0.15*1.01=0.1515
# If you make this adjustment and re-run the model you should see a final objective function value of $856,197.10





Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 34 rows, 20 columns and 139 nonzeros
Model fingerprint: 0x8520380e
Coefficient statistics:
  Matrix range     [5e-04, 1e+00]
  Objective range  [3e-01, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+04, 2e+05]
Presolve removed 8 rows and 1 columns
Presolve time: 0.01s
Presolved: 26 rows, 19 columns, 119 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.4907037e+06   1.164668e+05   0.000000e+00      0s
       7    8.0991139e+05   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.01 seconds (0.00 work units)
Optimal objective  8.099113856e+05


In [10]:
# Collect constraint attributes
constrs = m.getConstrs()
names  = m.getAttr(GRB.Attr.ConstrName, constrs)
pis    = m.getAttr(GRB.Attr.Pi,         constrs)
slacks = m.getAttr(GRB.Attr.Slack,      constrs)
rhs    = m.getAttr(GRB.Attr.RHS,        constrs)
low    = m.getAttr(GRB.Attr.SARHSLow,   constrs)
up     = m.getAttr(GRB.Attr.SARHSUp,    constrs)

def format_val(val):
    if abs(val) >= 1e8:
        return "inf"
    elif abs(val) < 1e-6:
        return "0"
    else:
        return f"{val:,.4f}"

# Header
print(f"{'Constraint':30s} {'Pi':>10s} {'Slack':>10s} {'RHS':>10s} {'Allow↓':>15s} {'Allow↑':>15s}")
print("-"*90)

# Body
for n, pi, s, r, lo, hi in zip(names, pis, slacks, rhs, low, up):
    allow_dec = r - lo
    allow_inc = hi - r
    print(f"{n:30s} {pi:10.4f} {s:10.4f} {r:10.4f} {format_val(allow_dec):>15s} {format_val(allow_inc):>15s}")

Constraint                             Pi      Slack        RHS          Allow↓          Allow↑
------------------------------------------------------------------------------------------
leq_constraints[Sugar Level 1]     0.0000    43.1579     0.0000         43.1579             inf
leq_constraints[Sugar Level 2] 13910.8705     0.0000     0.0000               0               0
leq_constraints[Sugar Level 3] 13910.8705     0.0000     0.0000               0               0
leq_constraints[Sugar Level 4]     0.0000    81.1837     0.0000         81.1837             inf
leq_constraints[Acidity 1]         0.0000 18684.2105     0.0000     18,684.2105             inf
leq_constraints[Acidity 2]         0.0000     0.0000     0.0000               0             inf
leq_constraints[Acidity 3]         0.0000     0.0000     0.0000               0             inf
leq_constraints[Acidity 4]         0.0000  8938.7755     0.0000      8,938.7755             inf
leq_constraints[Acidity 5]         0.0000    

In [11]:
m.printAttr('X')


    Variable            X 
-------------------------
wine_type_used[Va1]        50000 
wine_type_used[Va4]      2631.58 
wine_type_used[N2]        60000 
wine_type_used[N3]      9795.92 
wine_type_used[N4]      23265.3 
wine_type_used[M3]      20204.1 
wine_type_used[M4]       101020 


In [12]:
# TO FIND TOTAL QUANTITY OF EACH FINAL WINE (vintage_cabernet_a, vintage_cabernet_b, vintage_cabernet_c, N, M)
print("Total Quantity of Each Final Wine Blend:\n")

# Define base wine labels (prefixes)
wine_labels = ['Va', 'Vb', 'Vc', 'N', 'M']

for label in wine_labels:
    components = [w for w in winetype if w.startswith(label)]
    total_quantity = sum(blend_allocation[w].x for w in components)
    total_profit = sum(blend_allocation[w].x * blend_allocation[w].Obj for w in components)
    print(f"{label}: Quantity = {total_quantity:.2f}, Profit Contribution = {total_profit:.2f}")


Total Quantity of Each Final Wine Blend:

Va: Quantity = 52631.58, Profit Contribution = 352105.26
Vb: Quantity = 0.00, Profit Contribution = 0.00
Vc: Quantity = 0.00, Profit Contribution = 0.00
N: Quantity = 93061.22, Profit Contribution = 299204.08
M: Quantity = 121224.49, Profit Contribution = 158602.04


### 

In [13]:
## Discrete Commercial Decisions

# Binary variables are introduced only where economically meaningful, such as modeling quantity discounts or exclusive production choices.

# This allows the model to capture real business logic without unnecessary complexity.

# Commercial logic: binary switch to apply an all-units discount when threshold conditions are met.

# New binary variable to indicate if the discount applies
apply_discount = m.addVar(vtype=GRB.BINARY, name="apply_discount")

# Calculate the total Merlot 2010 used across all wines (vintage_cabernet_a_blend4, nonvintage_cabernet_blend4, nonvintage_merlot_blend4)
total_merlot_used = quicksum(blend_allocation[wine] for wine in ['vintage_cabernet_a_blend4', 'nonvintage_cabernet_blend4', 'nonvintage_merlot_blend4'])

# Add constraint: If total_merlot_used > 150,000, apply_discount should be 1
m.addConstr(total_merlot_used >= 150000 * apply_discount, "Apply_Discount")

# Add constraint: If total_merlot_used <= 150,000, apply_discount should be 0
m.addConstr(total_merlot_used <= 150000 + (1 - apply_discount) * 1e6, "No_Discount")

# Calculate the total savings from the discount, which is 0.45 (1.55 - 1.10) for each unit of Merlot when the discount applies
savings_merlot = 0.45 * total_merlot_used * apply_discount

# Calculate the total revenue (profit contribution already has the regular cost built in)
revenue = quicksum(
    profit_contribution[wine] * blend_allocation[wine] for wine in winetype
)

# Set the objective to maximize net profit (revenue minus cost of Merlot)
m.setObjective(revenue + savings_merlot, GRB.MAXIMIZE)

m.optimize() 

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 36 rows, 21 columns and 147 nonzeros
Model fingerprint: 0xbad82260
Model has 3 quadratic objective terms
Variable types: 20 continuous, 1 integer (1 binary)
Coefficient statistics:
  Matrix range     [5e-04, 1e+06]
  Objective range  [3e-01, 7e+00]
  QObjective range [9e-01, 9e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+04, 1e+06]
Found heuristic solution: objective -0.0000000
Presolve removed 15 rows and 0 columns
Presolve time: 0.00s
Presolved: 24 rows, 24 columns, 98 nonzeros
Found heuristic solution: objective 465382.35294
Variable types: 23 continuous, 1 integer (1 binary)

Root relaxation: objective 8.670242e+05, 14 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current 

In [14]:
m.printAttr('X')


    Variable            X 
-------------------------
wine_type_used[Va1]      38956.8 
wine_type_used[Va4]      2050.36 
wine_type_used[N1]      11043.2 
wine_type_used[N2]        60000 
wine_type_used[N3]      5513.87 
wine_type_used[N4]        25519 
wine_type_used[M3]      24486.1 
wine_type_used[M4]       122431 
apply_discount            1 


In [15]:
# TO FIND TOTAL QUANTITY OF EACH FINAL WINE (vintage_cabernet_a, vintage_cabernet_b, vintage_cabernet_c, N, M)
print("Total Quantity of Each Final Wine Blend:\n")

# Define base wine labels (prefixes)
wine_labels = ['Va', 'Vb', 'Vc', 'N', 'M']

for label in wine_labels:
    components = [w for w in winetype if w.startswith(label)]
    total_quantity = sum(blend_allocation[w].x for w in components)
    total_profit = sum(blend_allocation[w].x * blend_allocation[w].Obj for w in components)
    print(f"{label}: Quantity = {total_quantity:.2f}, Profit Contribution = {total_profit:.2f}")


Total Quantity of Each Final Wine Blend:

Va: Quantity = 41007.19, Profit Contribution = 274338.13
Vb: Quantity = 0.00, Profit Contribution = 0.00
Vc: Quantity = 0.00, Profit Contribution = 0.00
N: Quantity = 102076.05, Profit Contribution = 328333.25
M: Quantity = 146916.75, Profit Contribution = 192216.08


## Results & Interpretation

The optimized solution identifies which products to produce, at what scale, and how grape inventories should be allocated to maximize profit while respecting all constraints.

Rather than focusing on individual variable values, the key insight lies in which constraints bind and how they shape the optimal product mix.

In [16]:
# Strategic production choice: binary variables allow turning vintage/non-vintage production on/off under business rules.
# Define binary variables to enforce the either-or constraint
produce_vintage = m.addVar(vtype=GRB.BINARY, name="produce_vintage")
produce_non_vintage = m.addVar(vtype=GRB.BINARY, name="produce_non_vintage")

# Add constraint: You can only produce either Vintage or Non-Vintage Cabernet Sauvignon, not both
m.addConstr(produce_vintage + produce_non_vintage == 1, "Either_Vintage_Or_NonVintage")

# Add constraints to enforce the choice
# If producing Vintage, Non-Vintage Cabernet Sauvignon variables (N) must be zero
m.addConstr(quicksum(blend_allocation[wine] for wine in ['nonvintage_cabernet_blend1', 'nonvintage_cabernet_blend2', 'nonvintage_cabernet_blend3', 'nonvintage_cabernet_blend4']) <= 1e6 * produce_non_vintage, "NonVintage_Limit")

# If producing Non-Vintage, Vintage Cabernet Sauvignon variables (Va, Vb) must be zero
m.addConstr(quicksum(blend_allocation[wine] for wine in ['vintage_cabernet_a_blend1', 'vintage_cabernet_a_blend2', 'vintage_cabernet_a_blend3', 'vintage_cabernet_a_blend4', 'vintage_cabernet_b_blend1', 'vintage_cabernet_b_blend2', 'vintage_cabernet_b_blend3', 'vintage_cabernet_b_blend4']) <= 1e6 * produce_vintage, "Vintage_Limit")

# Objective function (same as before)
revenue = quicksum(
    profit_contribution[wine] * blend_allocation[wine] for wine in winetype
)
m.setObjective(revenue, GRB.MAXIMIZE)

m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 39 rows, 23 columns and 163 nonzeros
Model fingerprint: 0xb13e255c
Variable types: 20 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [5e-04, 1e+06]
  Objective range  [3e-01, 7e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+06]

MIP start from previous solve produced solution with objective 687214 (0.01s)
Loaded MIP start from previous solve with objective 687214

Presolve removed 18 rows and 2 columns
Presolve time: 0.00s
Presolved: 21 rows, 21 columns, 95 nonzeros
Variable types: 20 continuous, 1 integer (1 binary)

Root relaxation: objective 7.155880e+05, 16 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bound

### Solve the optimization model and extract the optimal solution

In [17]:
m.printAttr('X')


    Variable            X 
-------------------------
wine_type_used[N1]        50000 
wine_type_used[N2]        60000 
wine_type_used[N4]      36666.7 
wine_type_used[M3]        30000 
wine_type_used[M4]       150000 
produce_non_vintage            1 


In [18]:
# TO FIND TOTAL QUANTITY OF EACH FINAL WINE (vintage_cabernet_a, vintage_cabernet_b, vintage_cabernet_c, N, M)
print("Total Quantity of Each Final Wine Blend:\n")

# Define base wine labels (prefixes)
wine_labels = ['Va', 'Vb', 'Vc', 'N', 'M']

for label in wine_labels:
    components = [w for w in winetype if w.startswith(label)]
    total_quantity = sum(blend_allocation[w].x for w in components)
    total_profit = sum(blend_allocation[w].x * blend_allocation[w].Obj for w in components)
    print(f"{label}: Quantity = {total_quantity:.2f}, Profit Contribution = {total_profit:.2f}")


Total Quantity of Each Final Wine Blend:

Va: Quantity = 0.00, Profit Contribution = 0.00
Vb: Quantity = 0.00, Profit Contribution = 0.00
Vc: Quantity = 0.00, Profit Contribution = 0.00
N: Quantity = 146666.67, Profit Contribution = 476333.33
M: Quantity = 180000.00, Profit Contribution = 235500.00


In [ ]:
## Closing Note

This model is designed to support structured decision-making rather than provide a single static answer.

Its value lies in clarifying tradeoffs, identifying binding constraints, and enabling scenario analysis before committing operational decisions.